Tunable TIA design example with Evolutionary Optimization (Nevergrad) as a constraint satisfaction problem.

# Pre-body

## Clearing past runs (optional)

In [ ]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [ ]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

## Library Imports

In [ ]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.optimization.bayesian_ax    import Ax_Spice_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


## Loading the project config

In [ ]:
# ----------------------------
# Instantiations
# ----------------------------
ws_root = "/foss/designs/eda/SymXplorer/examples/tunable-tia"
pdk_name = "ihp-sg13g2"
yaml_file_name = "project_setup"
project_setup_yaml = Path(f"{ws_root}/{pdk_name}/spice/{yaml_file_name}.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

## Create a SPICE simulator wrapper

In [ ]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

## Create an optimizer object

In [ ]:
circuit_optimizer = Ax_Spice_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

## Sanity Check

In [ ]:
# wrapper.run_sanity_check(
#     use_editor=True,
#     sim_execution_t=Sim_Execution_Type.RUN_NOW
# )

# Main Body

## Optimization

In [ ]:
circuit_optimizer.parameterize()

In [ ]:
circuit_optimizer.optimize()

In [ ]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

## Inspection & Visualization

### (1) Best Param

In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

In [ ]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

In [ ]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

### (3) Metric Trace

In [ ]:
circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='gain_db', show=True)

In [ ]:
circuit_optimizer.plot_loss_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="fc", show=True)

### (4) Design Space Exploration

In [ ]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_ind_size", show=True)

# Testing

## Loss Function Testing

In [ ]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

In [ ]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

gain_db_min  = np.float64(-135.0)
gain_db_max  = np.float64(135)
points_per_unit = 10
gain_db_vals = np.linspace(gain_db_min, gain_db_max, int((gain_db_max-gain_db_min) * (points_per_unit)))  

for gain_db in gain_db_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'gain_db' : gain_db})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='gain_db', show = True)

In [ ]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

fc_min  = 4
fc_max  = 10
points_per_unit = 1000

# num_points = int((fc_max-fc_min) * (points_per_unit/1e3))
logger.info(f"using {points_per_unit} points")
logger.info(f"\tTarget: {PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name("fc")}")
# fc_vals = np.logspace(fc_min, fc_max)  
fc_vals = np.linspace(1e1, 1e8, 1000)  

for fc in fc_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'fc' : fc})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='fc', show = True)

## Other

In [ ]:
circuit_optimizer.optimization_log

In [ ]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

In [ ]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

In [ ]:
circuit_optimizer.compute_spec_loss(curr_val=-90, target_spec=target_spec)

In [ ]:
PROJECT_SETUP.dut_params